# Agent Bricks — DAIS 2026 Runbook

Story: Casper's Kitchens has hundreds of documents — inspection reports, legal complaints, regulatory filings, menus, audit reports. Unstructured, un-queryable, scattered. Agent Bricks turns them into a conversational intelligence layer on top of the operational data estate.

**Message:** Parse documents with SQL, make them conversational with Knowledge Assistants, route everything with a Supervisor — one platform, all governed by Unity Catalog.

In [ ]:
# Pre-flight: print fresh URLs for the demo.
# Re-run any time IDs go stale (e.g. after redeploy).

def _autodetect_caspers_catalog():
    """Find the most recently-deployed Casper's catalog via its uc_state table.

    Every Casper's deployment creates `<catalog>._internal_state.resources`,
    so the catalog with the most recently altered such table is the freshest
    deployment in this workspace.  Falls back to `caspersdev` if the system
    tables aren't queryable or no Casper's deployment is found.
    """
    try:
        rows = spark.sql("""
            SELECT table_catalog FROM system.information_schema.tables
            WHERE table_schema = '_internal_state' AND table_name = 'resources'
            ORDER BY last_altered DESC LIMIT 1
        """).collect()
        return rows[0].table_catalog if rows else "caspersdev"
    except Exception:
        return "caspersdev"

try:
    _detected = _autodetect_caspers_catalog()
    # Recreate the widget so the displayed default always reflects the freshest
    # deployment — `dbutils.widgets.text` does not reliably update the displayed
    # value when the widget already exists from a previous run.
    try:
        dbutils.widgets.remove("CATALOG")
    except Exception:
        pass
    dbutils.widgets.text("CATALOG", _detected, "UC Catalog")
    CATALOG = dbutils.widgets.get("CATALOG") or _detected
except Exception:
    CATALOG = "caspersdev"

import json
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
host = w.config.host.rstrip("/")

print(f"Catalog: {CATALOG}")
print(f"Host:    {host}\n")

# Knowledge Assistants
print("Knowledge Assistants")
print("-" * 80)
try:
    df = spark.sql(f"SELECT resource_data FROM {CATALOG}._internal_state.resources WHERE resource_type = 'knowledge_assistants' ORDER BY created_at ASC")
    # Defensive dedupe by KA tile_id — uc_state.add() doesn't enforce uniqueness,
    # so re-runs of the Knowledge_Agents stage can leave multiple rows per KA.
    seen_ids = set()
    for row in df.collect():
        info = json.loads(row.resource_data)
        tile_id = info.get("tile_id") or info.get("id") or info.get("endpoint_name")
        if tile_id and tile_id in seen_ids:
            continue
        seen_ids.add(tile_id)
        name = info.get("display_name") or info.get("name", "?")
        ep   = info.get("endpoint_name", "?")
        print(f"  {name}")
        print(f"    {host}/ml/endpoints/{ep}")
except Exception as e:
    print(f"  Could not read KAs from uc_state: {e}")

# Supervisor
print("\nMulti-Agent Supervisor")
print("-" * 80)
try:
    df2 = spark.sql(f"SELECT resource_data FROM {CATALOG}._internal_state.resources WHERE resource_type = 'multi_agent_supervisors' ORDER BY created_at DESC LIMIT 1")
    for row in df2.collect():
        info = json.loads(row.resource_data)
        ep = info.get("endpoint_name") or info.get("name", "?")
        print(f"  {ep}")
        print(f"  {host}/ml/endpoints/{ep}")
except Exception as e:
    print(f"  Could not read supervisor from uc_state: {e}")

# AI SQL tables
print("\nAI SQL tables")
print("-" * 80)
for tbl in ["ai_parsed_inspections", "ai_classified_inspections",
            "ai_extracted_inspections", "ai_summarized_inspections"]:
    try:
        cnt = spark.sql(f"SELECT COUNT(*) AS n FROM {CATALOG}.food_safety.{tbl}").collect()[0].n
        print(f"  {CATALOG}.food_safety.{tbl}  ({cnt} rows)")
    except Exception:
        print(f"  {CATALOG}.food_safety.{tbl}  (not found)")


### IPD: documents become data

![Programmable Document Pipelines — ai_parse_document, ai_query, ai_extract, ai_translate](assets/idp_demo.png)

| Function | What you get |
|---|---|
| `ai_parse_document` | Full text + structured VARIANT per PDF → `food_safety.ai_parsed_inspections` |
| `ai_classify` | Overall risk bucket + dominant violation category → `food_safety.ai_classified_inspections` |
| `ai_extract` | Structured fields (date, location, score, grade) → `food_safety.ai_extracted_inspections` |
| `ai_summarize` | 60-word plain-English summary → `food_safety.ai_summarized_inspections` |



In DBSQL or notebook: 

```sql
SELECT pdf_name, summary FROM <catalog>.food_safety.ai_summarized_inspections
```

`ai_classify` live:

```sql
SELECT pdf_name,
       ai_classify(summary, ARRAY('low', 'medium', 'high', 'critical')) AS risk
FROM <catalog>.food_safety.ai_summarized_inspections
WHERE pdf_name IS NOT NULL AND summary IS NOT NULL
LIMIT 5
```

The message: no pipeline, no custom model, no ETL framework. The AI is a SQL function. Any analyst who can write SQL can do this.
Code in `Environment_Helpers` task


### Knowledge Assistants — ask the documents

**Inspection KA:**

- **What were the critical violations at the worst-scoring location in the last inspection?**
- **Which location has the most repeat violations across all inspections?**
- **What corrective actions are overdue based on the latest reports?**

**Legal KA:**

- **Summarize the open vendor dispute complaints**
- **Are there any employment complaints that mention wage violations?**

**Regulatory KA:**

- **Which locations have permits expiring in the next 90 days?**
- **Are there any outstanding fire safety compliance items?**

The message: same chat experience as Genie, same UC governance — but the source is a PDF in a Volume, not a table.

### Supervisor Agent — route everything


![Supervisor Agent Architecture](assets/SA_architecture.png)

Open the supervisor endpoint. It routes across all 3 Genie spaces (structured data) and all 6 KAs (documents) automatically.

- **Give me the executive briefing on the state of the business.**

Show the routing — which sub-agents it called, in what order. Genie for revenue/ops metrics, KAs for inspection and compliance context.

- **Which location is the highest combined legal and operational risk right now?**

Needs both KA (legal complaints, inspection reports) and Genie (revenue trend, cancellation rate). Supervisor picks the right agents.

- **What are the top 3 things the CEO should know before the board call tomorrow?**
- **Is there anything in our regulatory filings that conflicts with how we're currently operating?**

Pure document reasoning — supervisor routes to Regulatory and Legal KAs.

Follow-up (same conversation):
- **Draft a 3-bullet action plan for the riskiest location.**

### MLflow experiment & evaluation

Every agent call is traced in MLflow — open the experiment from the Experiments sidebar. Each trace shows the full reasoning chain: which sub-agents were called, what they returned, latency per hop.

The `Evaluation` task ran a set of benchmark questions against the supervisor and scored the answers. Open the experiment → find the eval run → see pass rates and per-question results. This is how you know the agent is actually good before you put it in front of a customer.

Show `mas_XXX` experiment in MLflow

### What else Agent Bricks can do

- **Custom tools** — register any UC Function as a tool; the supervisor calls it when needed
- **Governed** — every sub-agent call runs under the caller's UC identity; all data access is audited

Source of truth: `stages/knowledge_agents.ipynb`, `stages/operational_supervisor.ipynb`, `stages/environment_helpers.ipynb`.